In [ ]:
# 1NNs-1types-convnext submit
# Inputs: isic-2024-challenge, tomoon33/{src,configs,pip},
#         kyunghyunmin/{train-all-data-0821-..., 260714-1nns-...}

SRC = "/kaggle/input/datasets/tomoon33/isic2024-src"
CONFIGS = "/kaggle/input/datasets/tomoon33/isic2024-configs"
PIP = "/kaggle/input/datasets/tomoon33/isic2024-pip"
GBDT = "/kaggle/input/datasets/kyunghyunmin/260714-1nns-1types-convnext-fev7-s5-tuning-weights"
DNN = "/kaggle/input/datasets/kyunghyunmin/train-all-data-0821-convnextv2-tiny-meta-t03-tsgkf"
COMP = "/kaggle/input/competitions/isic-2024-challenge"

open("/kaggle/working/_input_pip.txt", "w").write(PIP)

!rm -rf /kaggle/working/src /kaggle/working/isic2024-src
!cp -r {SRC} /kaggle/working/isic2024-src
!mv /kaggle/working/isic2024-src /kaggle/working/src

!rm -rf /kaggle/working/configs
!cp -r {CONFIGS} /kaggle/working/configs
!cp {GBDT}/260714-1NNs-1types-convnext-feV7-s5-tuning_weights.yaml /kaggle/working/configs/gbdt_params/

!mkdir -p /kaggle/working/logs/train_all_data/runs
!rm -f /kaggle/working/logs/train_all_data/runs/0821-convnextv2_tiny-meta_target03-transV2-lr1e-3-target_decay001-bs128_2-ep100-neg5-tsgkf
!ln -s {DNN} /kaggle/working/logs/train_all_data/runs/0821-convnextv2_tiny-meta_target03-transV2-lr1e-3-target_decay001-bs128_2-ep100-neg5-tsgkf

!mkdir -p /kaggle/working/logs/gbdt/runs
!rm -f /kaggle/working/logs/gbdt/runs/260714-1NNs-1types-convnext-feV7-s5-tuning_weights
!ln -s {GBDT} /kaggle/working/logs/gbdt/runs/260714-1NNs-1types-convnext-feV7-s5-tuning_weights

!touch /kaggle/working/.project-root
!mkdir -p /kaggle/working/data
!rm -f /kaggle/working/data/isic-2024-challenge
!ln -s {COMP} /kaggle/working/data/isic-2024-challenge

In [ ]:
gbdt_params = "260714-1NNs-1types-convnext-feV7-s5-tuning_weights"
batch_size_pred = 128
DEBUG_WITH_TRAIN_DATA = False
average = "simple"
use_model_dnn = "all_data"
use_model_gbdt = "all_data"
use_shina_models = False

assert average in ["simple", "rank"]
assert use_model_dnn in ["kfold", "all_data"]
assert use_model_gbdt in ["kfold", "all_data"]

In [ ]:
# Kaggle에 torch는 이미 있음 → torch/torchvision/xformers는 건너뛰고 나머지 설치
from pathlib import Path

PIP = "/kaggle/input/datasets/tomoon33/isic2024-pip"
print("PIP =", PIP)

skip = ("torch==", "torchvision==", "xformers==", "--")
lines = []
for line in Path(f"{PIP}/requirements.txt").read_text().splitlines():
    s = line.strip()
    if not s or s.startswith("#") or s.startswith(skip):
        continue
    lines.append(s)
Path("/kaggle/working/requirements_kaggle.txt").write_text("\n".join(lines) + "\n")

!pip install --no-index --find-links={PIP}/packages -r /kaggle/working/requirements_kaggle.txt
!python -c "import hydra, rootutils, lightning; print('ok', hydra.__version__)"

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

import hydra
import rootutils
import joblib
import os
from lightning import LightningDataModule, LightningModule, Trainer
from hydra import compose, initialize
from pathlib import Path
from glob import glob
import numpy as np
import torch
import pandas as pd
import lightgbm as lgb
import catboost as cb
import xgboost as xgb
from sklearn.linear_model import LogisticRegression
os.chdir("/kaggle/working/src")

rootutils.setup_root(Path().resolve(), indicator=".project-root", pythonpath=True)

from src.utils import RankedLogger
from src.isic_utils.feature_engineering_for_stacking import feature_engineering_for_stacking
from src.isic_utils.utils import comp_score, preprocess_df
from src.isic_utils.feature_engineering import feature_engineering_new
from src.isic_utils.gbdt_models import GBDTModels

root = os.environ["PROJECT_ROOT"]
log = RankedLogger(__name__, rank_zero_only=True)

In [ ]:
# load config
with initialize(version_base=None, config_path="configs"):
    cfg = compose(
        config_name="gbdt",
        overrides=[f"gbdt_params={gbdt_params}"],
        return_hydra_config=True,
    )
    cfg.paths.output_dir = "${hydra.runtime.output_dir}"
    cfg.paths.work_dir = "${hydra.runtime.cwd}"
    cfg.hydra.run.dir = cfg.log_dir
    cfg.hydra.runtime.output_dir = cfg.hydra.run.dir

if DEBUG_WITH_TRAIN_DATA:
    cfg.data.hdf5_test_name="train-image.hdf5"
    cfg.data.meta_csv_test_name="train-metadata.csv"
    
# prepare data
df_train = pd.read_csv(os.path.join(cfg.data.data_dir, cfg.data.meta_csv_train_name))
df_test = pd.read_csv(os.path.join(cfg.data.data_dir, cfg.data.meta_csv_test_name))

df_train, feature_cols, cat_cols = feature_engineering_new(df_train, version=cfg.gbdt_params.version_fe)
df_test, _, _ = feature_engineering_new(df_test, version=cfg.gbdt_params.version_fe)
df_train, df_test, feature_cols, cat_cols = preprocess_df(df_train, df_test, feature_cols, cat_cols)

In [ ]:
import gc

del df_train
gc.collect()

In [ ]:
import torch.distributed as dist
import torch.multiprocessing as mp
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, Dataset, DistributedSampler
import tempfile
from tqdm.notebook import tqdm
import gc

def setup(rank, world_size):
    os.environ['MASTER_ADDR'] = 'localhost'
    os.environ['MASTER_PORT'] = '12355'
    dist.init_process_group("gloo", rank=rank, world_size=world_size)

def cleanup():
    dist.destroy_process_group()

def inference(rank, world_size, temp_dir, cfg):
    print(f"Running inference on rank {rank}.")
    setup(rank, world_size)
    
    for dnn_run in cfg.gbdt_params.dnn_predictions:
        with initialize(version_base=None, config_path="configs"):
            cfg_dnn = compose(
                config_name="train",
                overrides=[f"experiment={dnn_run.name}"],
                return_hydra_config=True,
            )
            cfg_dnn.paths.output_dir = "${hydra.runtime.output_dir}"
            cfg_dnn.paths.work_dir =  "${hydra.runtime.cwd}"
            cfg_dnn.hydra.run.dir = cfg_dnn.log_dir
            cfg_dnn.hydra.runtime.output_dir = cfg_dnn.hydra.run.dir

        if cfg_dnn.model.net.get("pretrained"):
            cfg_dnn.model.net.pretrained=False
        if cfg_dnn.model.net.get("my_pretrain_path"):
            cfg_dnn.model.net.my_pretrain_path=None
        if cfg_dnn.model.net.get("image_model"):
            if cfg_dnn.model.net.image_model.get("pretrained"):
                cfg_dnn.model.net.image_model.pretrained=False
            if cfg_dnn.model.net.image_model.get("my_pretrain_path"):
                cfg_dnn.model.net.image_model.my_pretrain_path=None
        if cfg_dnn.model.net.get("meta_pretrain_path"):
            cfg_dnn.model.net.meta_pretrain_path=None
        if cfg_dnn.model.net.get("image_pretrain_path"):
            cfg_dnn.model.net.image_pretrain_path=None
                        
        if cfg_dnn.model.net.get("ckpt_path"):
            cfg_dnn.model.net.ckpt_path=None
        if cfg_dnn.model.net.get("encoder_image"):
            cfg_dnn.model.net.encoder_image.pretrained=False
            
        cfg_dnn.data.num_workers = 2
#         cfg_dnn.model.tta=tta
    
        if DEBUG_WITH_TRAIN_DATA:
            cfg_dnn.data.hdf5_test_name="train-image.hdf5"
            cfg_dnn.data.meta_csv_test_name="train-metadata.csv"

        datamodule: LightningDataModule = hydra.utils.instantiate(cfg_dnn.data)
        datamodule.setup("predict")
        dataset = datamodule.data_pred
        
        # 데이터셋과 데이터로더 설정
        sampler = DistributedSampler(dataset, num_replicas=world_size, rank=rank)
        dataloader = DataLoader(dataset, batch_size=batch_size_pred, sampler=sampler, num_workers=2)

        ckpt_paths = []
        for fold in range(cfg_dnn.data.n_fold):
            if fold == 0:
                ckpt_name = "last.ckpt"
            else:
                ckpt_name = f"last-v{fold}.ckpt"
            ckpt_paths.append(glob(os.path.join(cfg_dnn.paths.output_dir, "checkpoints", ckpt_name))[0])

        # 모델 설정
        model: LightningModule = hydra.utils.instantiate(cfg_dnn.model)
        model = model.to(rank).eval()
        if cfg_dnn.model.compile:
            model.net = torch.compile(model.net)

        for fold, ckpt_path in enumerate(ckpt_paths):
            model.load_state_dict(torch.load(ckpt_path, map_location="cpu", weights_only=False)["state_dict"])
        #     model = DDP(model.net, device_ids=[rank])
            net = model.net

            # 추론 실행
            net.eval()
            all_predictions = []
            ids = []
            with torch.inference_mode():
                with torch.cuda.amp.autocast():
                    for data in tqdm(dataloader, desc=f"rank_{rank}, fold_{fold}"):
                        if cfg_dnn.model._target_ == "src.models.isic2024_module_tip_finetune.ISIC2024LitModuleTIPFinetune":
                            logits = net(data["image"].to(rank), data["tabular"].to(rank))
                        else:
                            if cfg_dnn.model.get("use_image") and cfg_dnn.model.get("use_metadata"):
                                logits = net(data["image"].to(rank), data["metadata_num"].to(rank), data["metadata_cat"].to(rank))
                            else: #elif cfg.model.use_image:
                                logits = net(data["image"].to(rank))
                        
                        
                        if cfg.gbdt_params.use_logits:
                            preds = logits[:, 1]
                        else:
                            preds = torch.softmax(logits, dim=1)[:, 1]
                        all_predictions.append(preds.cpu())
                        ids.append(data["isic_id"])

            all_predictions = torch.cat(all_predictions)
            ids = np.concatenate(ids)

            # 추론 결과를 파일에 저장
            temp_file = os.path.join(temp_dir, f"predictions_{dnn_run.name}_rank_{rank}_fold_{fold}.pt")
            torch.save(all_predictions, temp_file)
            temp_file_id = os.path.join(temp_dir, f"ids_{dnn_run.name}_rank_{rank}_fold_{fold}.npy")
            np.save(temp_file_id, ids)

            print(f"{dnn_run.name}, rank {rank}, fold {fold} finished inference with predictions saved to {temp_file}, {temp_file_id}")
            
        del all_predictions, ids, datamodule, dataset, model, net, sampler, dataloader
        gc.collect()
    
    
def inference_mp(cfg):
    world_size = 2

    processes = []
    temp_dir = tempfile.mkdtemp()
    print(f"Temporary directory for storing predictions: {temp_dir}")
    for rank in range(world_size):
        p = mp.Process(
            target=inference, args=(rank, world_size, temp_dir, cfg)
        )
        p.start()
        processes.append(p)

    for p in processes:
        p.join()

    dnn_run_name_list = []
    df_dnn_preds_list = []
    for dnn_run in cfg.gbdt_params.dnn_predictions:
        df_preds_list = []
        for fold in range(cfg.data.n_fold):
            all_predictions = []
            all_ids = []
            for rank in range(world_size):
                temp_file = os.path.join(temp_dir, f"predictions_{dnn_run.name}_rank_{rank}_fold_{fold}.pt")
                predictions = torch.load(temp_file, weights_only=False)
                temp_file_id = os.path.join(temp_dir, f"ids_{dnn_run.name}_rank_{rank}_fold_{fold}.npy")
                ids = np.load(temp_file_id)
                all_predictions.append(predictions)
                all_ids.append(ids)

            # 결과를 하나의 텐서로 결합
            all_predictions = torch.cat(all_predictions)
            all_ids = np.concatenate(all_ids)
            df_preds = pd.DataFrame({"isic_id": all_ids, "target": all_predictions})
            df_preds = df_preds.drop_duplicates(subset=["isic_id"])
            df_preds_list.append(df_preds)
            
        df_dnn_preds_list.append(concat_df_preds(df_preds_list))
        dnn_run_name_list.append(dnn_run.name)
        
    return df_dnn_preds_list, dnn_run_name_list


def concat_df_preds(df_preds_list):
    df_preds = df_preds_list[0].rename({"target": "predictions"} ,axis="columns")
    for k, _df_preds in enumerate(df_preds_list[1:], start=1):
        df_preds = df_preds.merge(_df_preds.rename({"target": "predictions"} ,axis="columns"), how="left", on="isic_id", suffixes=("", f"_{k}"))
    df_preds = df_preds.rename({"predictions": "predictions_0"}, axis="columns")
    
    return df_preds

In [ ]:
def inference_all_data(rank, world_size, temp_dir, cfg):
    print(f"Running inference on rank {rank}.")
    setup(rank, world_size)
    
    for dnn_run in cfg.gbdt_params.dnn_predictions:
        with initialize(version_base=None, config_path="configs"):
            cfg_dnn = compose(
                config_name="train",
                overrides=[f"experiment={dnn_run.name}"],
                return_hydra_config=True,
            )
            cfg_dnn.task_name = "train_all_data"
            cfg_dnn.paths.output_dir = "${hydra.runtime.output_dir}"
            cfg_dnn.paths.work_dir =  "${hydra.runtime.cwd}"
            cfg_dnn.hydra.run.dir = cfg_dnn.log_dir
            cfg_dnn.hydra.runtime.output_dir = cfg_dnn.hydra.run.dir

        if cfg_dnn.model.net.get("pretrained"):
            cfg_dnn.model.net.pretrained=False
        if cfg_dnn.model.net.get("my_pretrain_path"):
            cfg_dnn.model.net.my_pretrain_path=None
        if cfg_dnn.model.net.get("image_model"):
            if cfg_dnn.model.net.image_model.get("pretrained"):
                cfg_dnn.model.net.image_model.pretrained=False
            if cfg_dnn.model.net.image_model.get("my_pretrain_path"):
                cfg_dnn.model.net.image_model.my_pretrain_path=None
        if cfg_dnn.model.net.get("meta_pretrain_path"):
            cfg_dnn.model.net.meta_pretrain_path=None
        if cfg_dnn.model.net.get("image_pretrain_path"):
            cfg_dnn.model.net.image_pretrain_path=None
                        
        if cfg_dnn.model.net.get("ckpt_path"):
            cfg_dnn.model.net.ckpt_path=None
        if cfg_dnn.model.net.get("encoder_image"):
            cfg_dnn.model.net.encoder_image.pretrained=False
            
        cfg_dnn.data.num_workers = 2
#         cfg_dnn.model.tta=tta
    
        if DEBUG_WITH_TRAIN_DATA:
            cfg_dnn.data.hdf5_test_name="train-image.hdf5"
            cfg_dnn.data.meta_csv_test_name="train-metadata.csv"

        datamodule: LightningDataModule = hydra.utils.instantiate(cfg_dnn.data)
        datamodule.setup("predict")
        dataset = datamodule.data_pred
        
        # 데이터셋과 데이터로더 설정
        sampler = DistributedSampler(dataset, num_replicas=world_size, rank=rank)
        dataloader = DataLoader(dataset, batch_size=batch_size_pred, sampler=sampler, num_workers=2)

        ckpt_name = "last.ckpt"
        print(os.path.join(cfg_dnn.paths.output_dir, "checkpoints", ckpt_name))
        ckpt_path = glob(os.path.join(cfg_dnn.paths.output_dir, "checkpoints", ckpt_name))[0]

        # 모델 설정
        model: LightningModule = hydra.utils.instantiate(cfg_dnn.model)
        model = model.to(rank).eval()
        if cfg_dnn.model.compile:
            model.net = torch.compile(model.net)

        model.load_state_dict(torch.load(ckpt_path, map_location="cpu", weights_only=False)["state_dict"])
    #     model = DDP(model.net, device_ids=[rank])
        net = model.net

        # 추론 실행
        net.eval()
        all_predictions = []
        ids = []
        with torch.inference_mode():
            with torch.cuda.amp.autocast():
                for data in tqdm(dataloader, desc=f"rank_{rank}"):
                    if cfg_dnn.model._target_ == "src.models.isic2024_module_tip_finetune.ISIC2024LitModuleTIPFinetune":
                        logits = net(data["image"].to(rank), data["tabular"].to(rank))
                    else:
                        if cfg_dnn.model.get("use_image") and cfg_dnn.model.get("use_metadata"):
                            logits = net(data["image"].to(rank), data["metadata_num"].to(rank), data["metadata_cat"].to(rank))
                        else: #elif cfg.model.use_image:
                            logits = net(data["image"].to(rank))


                    if cfg.gbdt_params.use_logits:
                        preds = logits[:, 1]
                    else:
                        preds = torch.softmax(logits, dim=1)[:, 1]
                    all_predictions.append(preds.cpu())
                    ids.append(data["isic_id"])

        all_predictions = torch.cat(all_predictions)
        ids = np.concatenate(ids)

        # 추론 결과를 파일에 저장
        temp_file = os.path.join(temp_dir, f"predictions_{dnn_run.name}_rank_{rank}.pt")
        torch.save(all_predictions, temp_file)
        temp_file_id = os.path.join(temp_dir, f"ids_{dnn_run.name}_rank_{rank}.npy")
        np.save(temp_file_id, ids)

        print(f"{dnn_run.name}, rank {rank} finished inference with predictions saved to {temp_file}, {temp_file_id}")
            
        del all_predictions, ids, datamodule, dataset, model, net, sampler, dataloader
        gc.collect()
    
    
def inference_mp_all_data(cfg):
    world_size = 2

    processes = []
    temp_dir = tempfile.mkdtemp()
    print(f"Temporary directory for storing predictions: {temp_dir}")
    for rank in range(world_size):
        p = mp.Process(
            target=inference_all_data, args=(rank, world_size, temp_dir, cfg)
        )
        p.start()
        processes.append(p)

    for p in processes:
        p.join()

    dnn_run_name_list = []
    df_dnn_preds_list = []
    for dnn_run in cfg.gbdt_params.dnn_predictions:
        all_predictions = []
        all_ids = []
        for rank in range(world_size):
            temp_file = os.path.join(temp_dir, f"predictions_{dnn_run.name}_rank_{rank}.pt")
            predictions = torch.load(temp_file, weights_only=False)
            temp_file_id = os.path.join(temp_dir, f"ids_{dnn_run.name}_rank_{rank}.npy")
            ids = np.load(temp_file_id)
            all_predictions.append(predictions)
            all_ids.append(ids)

        # 결과를 하나의 텐서로 결합
        all_predictions = torch.cat(all_predictions)
        all_ids = np.concatenate(all_ids)
        df_preds = pd.DataFrame({"isic_id": all_ids, "target": all_predictions})
        df_preds = df_preds.drop_duplicates(subset=["isic_id"])
        df_preds = df_preds.rename({"target": dnn_run.name} ,axis="columns")

        df_dnn_preds_list.append(df_preds)
        dnn_run_name_list.append(dnn_run.name)
        
    return df_dnn_preds_list, dnn_run_name_list

In [ ]:
if use_model_dnn == "kfold":
    df_dnn_preds_list, dnn_run_name_list = inference_mp(cfg)
    # fold 평균 계산
    for run_name, df_preds in zip(dnn_run_name_list, df_dnn_preds_list):
        df_preds[run_name] = df_preds[['predictions_0', 'predictions_1', 'predictions_2', 'predictions_3', 'predictions_4']].mean(1)
        df_test = df_test.merge(df_preds[["isic_id", run_name]], how="left", on="isic_id")
        
elif use_model_dnn == "all_data":
    df_dnn_preds_list, dnn_run_name_list = inference_mp_all_data(cfg)
    for run_name, df_preds in zip(dnn_run_name_list, df_dnn_preds_list):
        df_test = df_test.merge(df_preds[["isic_id", run_name]], how="left", on="isic_id")
        
feature_cols += dnn_run_name_list

In [ ]:
# shina's models
if use_shina_models:
    !python /kaggle/input/edgenext-scripts/edgenext_fp16_mod.py 5 1
    !mv submission.csv submission_edgenext0822.csv

    !python /kaggle/input/edgenext-scripts/edgenext_128_fp16_mod.py 5 1
    !mv submission.csv submission_edgenext0822_128.csv

    df_preds = pd.read_csv("submission_edgenext0822.csv")
    df_preds = df_preds[["isic_id", "target"]].rename({"target": "edgenext0822"}, axis=1)
    df_test = df_test.merge(df_preds, how="left", on="isic_id")

    df_preds = pd.read_csv("submission_edgenext0822_128.csv")
    df_preds = df_preds[["isic_id", "target"]].rename({"target": "edgenext0822_128"}, axis=1)
    df_test = df_test.merge(df_preds, how="left", on="isic_id")

    feature_cols += ["edgenext0822", "edgenext0822_128"]

In [ ]:
# DBGT prediction
if use_model_gbdt == "kfold":
    predictions_list = []
    for fold in range(cfg.data.n_fold):
        save_file_path = os.path.join(cfg.log_dir, f"model_{fold}.joblib")
        model = joblib.load(save_file_path)

        preds = model.predict(df_test)
        predictions_list.append(preds)
        
        del model
        gc.collect()

    predictions_list = np.stack(predictions_list)

    if average == "rank":
        df_rank = pd.DataFrame(predictions_list.T).rank()
        n_preds = df_rank.shape[1]
        df_rank["rank_sum"] = np.sum(df_rank[col] for col in df_rank.columns)
        df_rank["target"] = df_rank["rank_sum"] / (n_preds * df_rank.shape[0])
        target = df_rank["target"].values
    elif average == "simple":
        target = predictions_list.mean(0)
    
elif use_model_gbdt == "all_data":
    save_file_path = os.path.join(cfg.log_dir, f"model_all_data.joblib")
    model = joblib.load(save_file_path)
    target = model.predict(df_test)
    
    del model
    gc.collect()
    

In [ ]:
assert not DEBUG_WITH_TRAIN_DATA
df_submit = pd.DataFrame({"isic_id":df_test["isic_id"],"target":target})
df_submit.to_csv("/kaggle/working/submission.csv", index=False)

In [ ]:
from IPython.display import display

display(df_submit.head())